**Author** Martín Gamboa

**Github** mmgamboa

**Date** January 10th, 2025

# Problem 1: Handling Outliers in Regression or Chi-Square Fitting

**Objective**. Explore different alternatives to determine whether an outlier should be considered or discarded when performing linear regression or chi-square fitting. Special attention is required for borderline cases where it is not evident if the point should be excluded.

**Requirements**

* Avoid using smoothness techniques.
* Effectiveness is not the priority; computational resource requirements for daily computation must be explicitly stated.

**Data**.
* QQQ and IWM (used as benchmarks).
* 2YM (likely referring to a 2-year metric or dataset).
* Use a logarithmic scale for computations.


In [15]:
# Get the data QQQ and IWM from Yahoo Finance
# Load specific packages
from importlib import reload
import time

import yfinance as yf

import numpy as np
import pandas as pd
import plotly.express as px

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# Import mymodule
import sys
sys.path.append('..')
from src.features.build_features import compute_daily_return
from src.models.libfit import find_closest_date,apply_filter_by_dates,fit_line, fit_adaptative_line

# Get Data

In [ ]:
#! Close or Adj Close?
param = 'Adj Close'
# Download historical data for QQQ and IWM 2YW
#data = yf.download(['QQQ', 'IWM'], period='2y')[param]
data = pd.read_csv("../data/raw/IWM_QQQ_20240201_20250201.csv", 
                   header=[0, 1], index_col=0)

# Normalize data to start at 1
data = data / data.iloc[0]

In [5]:
data

Price      Adj Close               Close                High            \
Ticker           IWM       QQQ       IWM       QQQ       IWM       QQQ   
Date                                                                     
2024-02-01  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
2024-02-02  0.994730  1.016901  0.994730  1.016901  0.998774  1.018609   
2024-02-05  0.982092  1.015573  0.982092  1.015573  0.986564  1.017117   
2024-02-06  0.990944  1.013535  0.990943  1.013535  0.990089  1.018561   
2024-02-07  0.988641  1.023964  0.988641  1.023964  0.991877  1.024741   
...              ...       ...       ...       ...       ...       ...   
2025-01-27  1.172556  1.226097  1.158923  1.218854  1.173087  1.226360   
2025-01-28  1.173851  1.244219  1.160203  1.236868  1.162563  1.238221   
2025-01-29  1.171159  1.241882  1.157542  1.234545  1.165934  1.237251   
2025-01-30  1.183065  1.247175  1.169310  1.239807  1.175284  1.245561   
2025-01-31  1.172453  1.245363  1.158821  1.238006  1.176663  1.258393   

Price            Low                Open              Volume            
Ticker           IWM       QQQ       IWM       QQQ       IWM       QQQ  
Date                                                                    
2024-02-01  1.000000  1.000000  1.000000  1.000000  1.000000  1.000000  
2024-02-02  1.004490  1.011919  0.994493  1.011963  0.888129  1.168083  
2024-02-05  0.992325  1.017209  0.990531  1.024547  0.668335  0.782009  
2024-02-06  0.997337  1.017735  0.986568  1.025836  0.583216  0.702746  
2024-02-07  1.002663  1.026998  0.998765  1.027747  0.533195  0.739340  
...              ...       ...       ...       ...       ...       ...  
2025-01-27  1.175847  1.221010  1.166744  1.220182  0.461704  1.191401  
2025-01-28  1.175690  1.224911  1.166898  1.230259  0.283250  0.650757  
2025-01-29  1.172558  1.237165  1.166589  1.247547  0.340969  0.522441  
2025-01-30  1.184775  1.240301  1.176059  1.250531  0.429985  0.537778  
2025-01-31  1.177570  1.247433  1.177243  1.258196  0.594307  0.761548  

[251 rows x 12 columns]

In [7]:
# Get date using .index
dates = data.index
#dates = [d.strftime('%Y-%m-%d') for d in dates]

Display Log Return difference and Scatter plot to show any possible trend between prices

In [10]:
log_returns_difference = compute_daily_return(data['Adj Close'], data.index, ['QQQ', 'IWM'])

# Plot log returns difference with color line purple
fig = px.line(log_returns_difference, title='Log Returns Difference (QQQ - IWM)')
# Plot horizontal line 
fig.add_hline(y=0, line_dash="dot", line_color="red")
# Set x-label
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Log Returns Difference')
# Set line color to purple
fig.update_traces(line_color='purple')

fig.show()

In [11]:
log_returns_difference

,QQQ,IWM
Date,,
2024-02-01,0.000000,0.000000
2024-02-02,0.016759,-0.005284
2024-02-05,-0.001306,-0.012786
2024-02-06,-0.002009,0.008973
2024-02-07,0.010238,-0.002326
...,...,...
2025-01-27,-0.029547,-0.009622
2025-01-28,0.014672,0.001103
2025-01-29,-0.001880,-0.002296


In [12]:
xdata_label = 'IWM'
ydata_label = 'QQQ' 
fig = px.scatter(x=log_returns_difference[xdata_label], 
                 y=log_returns_difference[ydata_label])
fig.update_xaxes(title_text=xdata_label)
fig.update_yaxes(title_text=ydata_label)
fig.show()

# Outlier detection in datset

In [13]:
full_indexes = log_returns_difference.index.values

#print(find_closest_date('2023-02-19', pd.to_datetime(full_indexes)))

# Get date indices for the sliders
date_indices = {i: date for i, date in enumerate(dates)}

In [16]:
# Initialize Dash app
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Interactive Scatter Plot with Fitted Line"),
    
    dcc.Graph(id='scatter-plot'),
    
    html.Label("Threshold for Outlier Detection:"),
    dcc.Slider(id='threshold-slider', 
               min=1, 
               max=5, 
               step=0.5, 
               value=1.5,
               marks={i: str(i) for i in range(1, 6)}),
    html.Label("Select Initial Date:"),
    dcc.RangeSlider(id='date-range-slider', 
                    min=0, 
                    max=len(dates)-1, 
                    step=1, 
                    value=[0, len(dates)-1],
                    marks={i: date_indices[i] for i in range(0, len(dates), 30)}),
    html.Div([dcc.Dropdown(['std', 'iqr'], 
                            id='outlier-strategy', 
                            value='std',
                            ) ]),
    
])

@app.callback(
    Output('scatter-plot', 'figure'),
    Input('threshold-slider', 'value'),
    [Input('date-range-slider', 'value')],
    Input('outlier-strategy', 'value')
)

def update_plot(threshold, range_dates, outlier_strategy):
    # Convert slider indices to dates
    # Debugging: print the value of range_dates
    print('__________________________________________')
    t0 = time.time()
    initial_date = find_closest_date(pd.to_datetime(dates[range_dates[0]]), pd.to_datetime(full_indexes))
    end_date = find_closest_date(pd.to_datetime(dates[range_dates[1]]), pd.to_datetime(full_indexes))

    
    # Filter data based on the selected date range
    filtered_data = apply_filter_by_dates(log_returns_difference, initial_date, end_date)
    print("Removing outliers with method: ", outlier_strategy)
    
    ## Fit raw data
    X = filtered_data[xdata_label].values.reshape(-1, 1)
    y = filtered_data[ydata_label].values

    x_pred, y_pred, reg_model = fit_line(X, y, nvals=100)
    # Compute residuals
    residuals = y - reg_model.predict(X)
    
    # Fit line without outliers
    # Adaptative fitting. If an outlier is close enough to the line, it is considered an inlier
    x_pred_no_outliers, y_pred_no_outliers, accepted_idxs = fit_adaptative_line(X, y, residuals, 
                                                                                   initial_date, 
                                                                                   end_date, 
                                                                                   outlier_strategy, 
                                                                                   threshold)

    print(f"Time elapsed in preprocessed, outliers and fitting: {time.time() - t0}")
    
    print("Monitor resources pre-plotting", )
    # Create the figure
    fig = go.Figure()
    
    # Add a line trace
    fig.add_scatter(
        x=x_pred, 
        y=y_pred, 
        mode="markers", 
        line=dict(color="red"),
        name='Fitted line')

    
    # Raw data points
    fig.add_trace(go.Scatter(x=X.flatten(), y=y, mode='markers', 
                             name='Raw data', marker=dict(color='blue')))
    
    # Fitted line without outliers
    fig.add_trace(go.Scatter(x=x_pred_no_outliers, y=y_pred_no_outliers, mode='lines', 
                             name='Fitted line (no outliers)', line=dict(color='red')))
    
    # Outliers
    fig.add_trace(go.Scatter(x=X[~accepted_idxs].flatten(), y=y[~accepted_idxs], mode='markers', 
                             name='Outliers', marker=dict(color='black')))
    # Update axes labels
    fig.update_xaxes(title_text=xdata_label, range=[X.min()-np.abs(X.min())*0.1, 
                                                    X.max()+np.abs(X.max())*0.1])
    fig.update_yaxes(title_text=ydata_label, range=[y.min()-np.abs(y.min())*0.1, 
                                                    y.max()+np.abs(y.max())*0.1])

    # Update layout
    fig.update_layout(
        xaxis_title=xdata_label,
        yaxis_title=ydata_label,
        #title="Scatter Plot with Fitted Line",
        legend=dict(orientation="h", x=0, y=-0.2)
    )
    print("Monitor resources post-plotting", )
    return fig

app.run_server(mode='inline', debug=True)


__________________________________________
Removing outliers with method:  std
Score: 0.357272646
Coef: 0.54 - 0.00054
Fitting line from 2024-02-01 to 2025-01-31. ======================
Score: 0.483679911
Coef: 0.51 - 0.00110
Score: 0.459850170
Coef: 0.54 - 0.00106
Time elapsed in preprocessed, outliers and fitting: 0.016731977462768555
Monitor resources pre-plotting
Monitor resources post-plotting


In [17]:
# Resources requirements
print(f"Pandas DataFrame used {log_returns_difference.memory_usage(deep=True).sum()/1024**2:4.3f} MB")
    

Pandas DataFrame used 0.018 MB
